1. Load data and reproduce original feature engineering

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix
from xgboost import XGBClassifier

df = pd.read_csv("../data/fraudTrain.csv")
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
df["dob"] = pd.to_datetime(df["dob"])
df["hour"] = df["trans_date_trans_time"].dt.hour
df["day_of_week"] = df["trans_date_trans_time"].dt.day_name()
df["age"] = ((df["trans_date_trans_time"] - df["dob"]).dt.days / 365.25).astype(int)
R = 6371
lat1, lon1 = np.radians(df["lat"]), np.radians(df["long"])
lat2, lon2 = np.radians(df["merch_lat"]), np.radians(df["merch_long"])
dlat, dlon = lat2 - lat1, lon2 - lon1
a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
df["distance_km"] = 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
features = ["amt", "category", "hour", "day_of_week", "age", "distance_km"]
X, y = df[features], df["is_fraud"]
print("Dataset:", X.shape)

Dataset: (1296675, 6)


2. Recreate the same splits as 03_xgboost.ipynb

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.20, random_state=42, stratify=y_train
)
print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
assert len(X_train) == 829872 and len(X_val) == 207468 and len(X_test) == 259335

Train: (829872, 6) Validation: (207468, 6) Test: (259335, 6)


3. Prepare both feature sets 

In [3]:
def make_preprocessor(include_age):
    numeric = ["amt", "hour", "age", "distance_km"] if include_age else ["amt", "hour", "distance_km"]
    return ColumnTransformer(transformers=[
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["category", "day_of_week"])
    ])

X_train_no_age = X_train.drop(columns="age")
X_val_no_age = X_val.drop(columns="age")
preprocessor_with_age = make_preprocessor(True)
preprocessor_no_age = make_preprocessor(False)
train_with_age = preprocessor_with_age.fit_transform(X_train)
val_with_age = preprocessor_with_age.transform(X_val)
train_no_age = preprocessor_no_age.fit_transform(X_train_no_age)
val_no_age = preprocessor_no_age.transform(X_val_no_age)
print("With age:", train_with_age.shape, "Without age:", train_no_age.shape)

With age: (829872, 25) Without age: (829872, 24)


4. Train both models with identical hyperparameters

In [4]:
def make_model():
    return XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05,
                         objective="binary:logistic", eval_metric="aucpr",
                         tree_method="hist", random_state=42)

model_with_age = make_model()
model_no_age = make_model()
model_with_age.fit(train_with_age, y_train)
model_no_age.fit(train_no_age, y_train)
print("Both models trained")

Both models trained


5. Evaluate on the same validation set at threshold 0.4

In [5]:
threshold = 0.4
prob_with_age = model_with_age.predict_proba(val_with_age)[:, 1]
prob_no_age = model_no_age.predict_proba(val_no_age)[:, 1]
pred_with_age = (prob_with_age >= threshold).astype(int)
pred_no_age = (prob_no_age >= threshold).astype(int)

def metrics(prob, pred):
    return {"Precision": precision_score(y_val, pred),
            "Recall": recall_score(y_val, pred),
            "F1": f1_score(y_val, pred),
            "PR-AUC": average_precision_score(y_val, prob)}

comparison = pd.DataFrame({
    "With Age": metrics(prob_with_age, pred_with_age),
    "Without Age": metrics(prob_no_age, pred_no_age)
})
comparison["Difference (without - with)"] = comparison["Without Age"] - comparison["With Age"]
display(comparison.round(4))
print("With age confusion matrix:\n", confusion_matrix(y_val, pred_with_age))
print("Without age confusion matrix:\n", confusion_matrix(y_val, pred_no_age))

,With Age,Without Age,Difference (without - with)
Precision,0.8936,0.8296,-0.0640
Recall,0.8185,0.7419,-0.0766
F1,0.8544,0.7833,-0.0711
PR-AUC,0.9097,0.8504,-0.0593


With age confusion matrix:
 [[206150    117]
 [   218    983]]
Without age confusion matrix:
 [[206084    183]
 [   310    891]]


6. Fairness Evaluation


In [6]:
age_bins = [0, 25, 40, 60, np.inf]
age_labels = ["Under 25", "25-39", "40-59", "60+"]

fairness_df = pd.DataFrame({
    "age": X_val["age"].to_numpy(),
    "actual": y_val.to_numpy(),
    "with_age": pred_with_age,
    "without_age": pred_no_age
})

fairness_df["age_group"] = pd.cut(
    fairness_df["age"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

results = []

for group_name, group in fairness_df.groupby(
    "age_group", observed=True
):
    legitimate = group[group["actual"] == 0]

    fpr_with = (legitimate["with_age"] == 1).mean()
    fpr_without = (legitimate["without_age"] == 1).mean()

    results.append({
        "Age Group": group_name,
        "Legitimate Transactions": len(legitimate),
        "FPR With Age": fpr_with,
        "FPR Without Age": fpr_without
    })

fairness_results = pd.DataFrame(results)

display(fairness_results)

,Age Group,Legitimate Transactions,FPR With Age,FPR Without Age
0,Under 25,19268,0.000675,0.001038
1,25-39,66011,0.000894,0.001091
2,40-59,77737,0.000515,0.000836
3,60+,43251,0.000116,0.000601


In [7]:
results = []

for group_name, group in fairness_df.groupby(
    "age_group", observed=True
):
    fraudulent = group[group["actual"] == 1]

    fnr_with = (fraudulent["with_age"] == 0).mean()
    fnr_without = (fraudulent["without_age"] == 0).mean()

    results.append({
        "Age Group": group_name,
        "Fraudulent Transactions": len(fraudulent),
        "FNR With Age": fnr_with,
        "FNR Without Age": fnr_without
    })

fnr_results = pd.DataFrame(results)

display(fnr_results)

,Age Group,Fraudulent Transactions,FNR With Age,FNR Without Age
0,Under 25,108,0.203704,0.212963
1,25-39,308,0.275974,0.344156
2,40-59,450,0.193333,0.266667
3,60+,335,0.071642,0.182090
